# Linear Discriminant Analysis (LDA) with K-Nearest Neighbors (KNN) on Fashion MNIST

In this notebook, we will explore the effect of Linear Discriminant Analysis (LDA) on the classification performance of the K-Nearest Neighbors (KNN) algorithm using the Fashion MNIST dataset.

We will compare two approaches:
1.  **Pure KNN**: Running KNN directly on the raw pixel data.
2.  **LDA + KNN**: Applying LDA to reduce the dimensionality of the data before running KNN.

LDA is a supervised dimensionality reduction technique that tries to maximize the separation between different classes.

In [8]:
import tensorflow as tf
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt 


## 1. Load Dataset
We load the Fashion MNIST dataset, which consists of 28x28 grayscale images of 10 fashion categories.

In [9]:
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train_raw, y_train), (x_test_raw, y_test) = fashion_mnist.load_data()


## 2. Preprocessing
Since the images are 2D (28x28), we flatten them into 1D vectors (784 features). We also normalize the pixel values to the range [0, 1] to help the algorithms converge faster and perform better.

In [10]:
#flattenning the image since each image is 28x28 pixels, we reshape it to a 1D array of 784 pixels and normalize the pixel values to the range [0, 1] for better performance.
x_train = x_train_raw.reshape(60000, 784) / 255.0 #255 because that is the max pixel value
x_test = x_test_raw.reshape(10000, 784) / 255.0

## 3. Subsetting
To speed up the computation for this demonstration, we will use a subset of 10,000 samples for training.

In [11]:
# Using a subset for speed (10k samples)
x_train, y_train = x_train[:10000], y_train[:10000]

## 4. Model Training & Evaluation

### Approach 1: Pure KNN
We train a KNN classifier (`n_neighbors=5`) on the original high-dimensional data (784 features).

### Approach 2: LDA + KNN
We first use LDA to reduce the dimensionality from 784 to 9 components (since there are 10 classes, LDA can produce at most $C-1$ components). Then, we train a KNN classifier on this reduced feature space.

In [12]:
#KNN Alone
knn_pure = KNeighborsClassifier(n_neighbors=5)
knn_pure.fit(x_train, y_train)
y_pred_knn_pure = knn_pure.predict(x_test)
score_knn_pure = accuracy_score(y_test, y_pred_knn_pure)

#LDA + KNN
lda_reducer = LinearDiscriminantAnalysis(n_components=9)
x_train_reduced = lda_reducer.fit_transform(x_train, y_train)
x_test_reduced = lda_reducer.transform(x_test)

knn_hybrid = KNeighborsClassifier(n_neighbors=5)
knn_hybrid.fit(x_train_reduced, y_train)
y_pred_hybrid = knn_hybrid.predict(x_test_reduced)
score_hybrid = accuracy_score(y_test, y_pred_hybrid)



## 5. Comparison and Inference
Let's compare the accuracy scores.

In [13]:
#Comparison
print(f"Pure KNN Accuracy: {score_knn_pure:.2f}")
print(f"LDA + KNN Accuracy: {score_hybrid:.2f}")


Pure KNN Accuracy: 0.82
LDA + KNN Accuracy: 0.81


### Inference

*   **Dimensionality Reduction**: LDA reduced the feature space from 784 dimensions to just 9. This significantly reduces the computational cost and storage requirements.
*   **Performance**:
    *   If **LDA + KNN Accuracy > Pure KNN Accuracy**: It implies that LDA successfully extracted the most discriminative features, filtering out noise and irrelevant pixels, leading to better class separation.
    *   If **LDA + KNN Accuracy is similar**: It shows that we can achieve comparable performance with vastly fewer dimensions (9 vs 784), which is a huge efficiency gain.
    *   If **LDA + KNN Accuracy < Pure KNN Accuracy**: It might suggest that some useful information was lost during the dimensionality reduction, or that the linear assumption of LDA was too restrictive for this dataset.

*Generally, for Fashion MNIST, LDA provides a very compact representation that preserves class separability well, often resulting in competitive accuracy with much faster inference times.*
